In [ ]:
from Lora import Lora
import torch
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer
import copy

# Load IMDb and keep the official test split out of training and validation.
dataset = load_dataset("stanfordnlp/imdb")
training_split = dataset["train"].shuffle(seed=42).train_test_split(test_size=0.1, seed=42)
train_data = training_split["train"]
validation_data = training_split["test"]
test_data = dataset["test"].shuffle(seed=42)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)


def tokenize(token_batch):
    return tokenizer(token_batch["text"], padding=True, truncation=True, max_length=512)


training_data = train_data.map(tokenize, batched=True)
validation_data = validation_data.map(tokenize, batched=True)
testing_data = test_data.map(tokenize, batched=True)

base_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
base_model_copied = copy.deepcopy(base_model)
lora_model = Lora(
    base_model_copied,
    r=64,
    alpha=128,
    dropout=0.1,
    target_weights=["key", "value"],
).update_module()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1121.47it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [30]:
trainable_params = sum(
    parameter.numel()
    for parameter in base_model.parameters()
    if parameter.requires_grad
)

trainable_params

109483778

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-bert-imdb",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=training_data,
    eval_dataset=validation_data,
)

In [20]:
trainable_params = sum(
    parameter.numel()
    for parameter in lora_model.parameters()
    if parameter.requires_grad
)

trainable_params

3538944

In [ ]:
trainer.train()

In [13]:
test_results = trainer.predict(testing_data)

In [14]:
from sklearn.metrics import fbeta_score
fbeta_score(test_results.label_ids, test_results.predictions.argmax(axis=-1), beta=1)

0.930225169770859

In [15]:
import pandas as pd
loss_history = [
    entry for entry in trainer.state.log_history
    if "loss" in entry
]

loss_df = pd.DataFrame(loss_history)
loss_df.to_csv("training_loss_key_value_query.csv", index=False)

In [4]:
for parameter in base_model.parameters():
    parameter.requires_grad = True

In [ ]:
base_training_args = copy.deepcopy(training_args)
base_training_args.output_dir = "./full-finetune-bert-imdb"
trainer_base = Trainer(
    model=base_model,
    args=base_training_args,
    train_dataset=training_data,
    eval_dataset=validation_data,
)

In [ ]:
trainer_base.train()

Step,Training Loss
10,0.666782
20,0.591544
30,0.503452
40,0.432912
50,0.415943
60,0.498954
70,0.389096
80,0.454102
90,0.334538
100,0.269910


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.41it/s]


TrainOutput(global_step=3126, training_loss=0.19080901078245843, metrics={'train_runtime': 829.5235, 'train_samples_per_second': 60.276, 'train_steps_per_second': 3.768, 'total_flos': 1.3155552768e+16, 'train_loss': 0.19080901078245843, 'epoch': 2.0})

In [ ]:
base_test_results = trainer_base.predict(testing_data)

In [ ]:
import pandas as pd

base_predictions = base_test_results.predictions.argmax(axis=-1)
base_f1 = fbeta_score(
    base_test_results.label_ids,
    base_predictions,
    beta=1,
    zero_division=0,
)
pd.DataFrame([{"f1_score": base_f1}]).to_csv(
    "full_finetune_f1_score.csv",
    index=False,
)

In [14]:
import pandas as pd
loss_history = [
    entry for entry in trainer_base.state.log_history
    if "loss" in entry
]

loss_df = pd.DataFrame(loss_history)
loss_df.to_csv("original_model.csv", index=False)

In [32]:
predicted_labels = test_results.predictions.argmax(axis=-1)
true_labels = test_results.label_ids

f1 = fbeta_score(
    true_labels,
    predicted_labels,
    beta=1,
    zero_division=0,
)

print(f"F1 score: {f1:.4f}")

F1 score: 0.9400


In [33]:
test_results.label_ids

array([1, 1, 0, ..., 0, 1, 0], shape=(25000,))

In [36]:
testing_data.select(range(10))

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 10
})

In [35]:
test_data[:10]

{'text': ["<br /><br />When I unsuspectedly rented A Thousand Acres, I thought I was in for an entertaining King Lear story and of course Michelle Pfeiffer was in it, so what could go wrong?<br /><br />Very quickly, however, I realized that this story was about A Thousand Other Things besides just Acres. I started crying and couldn't stop until long after the movie ended. Thank you Jane, Laura and Jocelyn, for bringing us such a wonderfully subtle and compassionate movie! Thank you cast, for being involved and portraying the characters with such depth and gentleness!<br /><br />I recognized the Angry sister; the Runaway sister and the sister in Denial. I recognized the Abusive Husband and why he was there and then the Father, oh oh the Father... all superbly played. I also recognized myself and this movie was an eye-opener, a relief, a chance to face my OWN truth and finally doing something about it. I truly hope A Thousand Acres has had the same effect on some others out there.<br /><

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)

lora_labels = test_results.label_ids
lora_predictions = test_results.predictions.argmax(axis=-1)
base_labels = base_test_results.label_ids
base_predictions = base_test_results.predictions.argmax(axis=-1)

metric_rows = []
for model_name, labels, predictions in [
    ("LoRA", lora_labels, lora_predictions),
    ("Full fine-tuning", base_labels, base_predictions),
]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0,
    )
    metric_rows.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(labels, predictions),
            "Precision": precision,
            "Recall": recall,
            "F1": f1,
        }
    )

metrics = pd.DataFrame(metric_rows).set_index("Model")
display(metrics.round(4))

print("LoRA classification report")
print(classification_report(lora_labels, lora_predictions, target_names=["negative", "positive"], zero_division=0))
print("Full fine-tuning classification report")
print(classification_report(base_labels, base_predictions, target_names=["negative", "positive"], zero_division=0))

fig, ax = plt.subplots(figsize=(7, 4))
metrics[["Accuracy", "Precision", "Recall", "F1"]].plot(
    kind="bar",
    ylim=(0, 1),
    color=["#287f8e", "#e29a50", "#719b6b", "#bd6653"],
    ax=ax,
)
ax.set_ylabel("Score")
ax.set_xlabel("")
ax.set_title("IMDb test-set performance")
ax.tick_params(axis="x", rotation=0)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
fig.savefig(results_dir / "metrics_comparison.png", dpi=160, bbox_inches="tight")
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, title, labels, predictions in [
    (axes[0], "LoRA", lora_labels, lora_predictions),
    (axes[1], "Full fine-tuning", base_labels, base_predictions),
]:
    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    ax.imshow(matrix, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks([0, 1], labels=["negative", "positive"])
    ax.set_yticks([0, 1], labels=["negative", "positive"])
    for row in range(2):
        for column in range(2):
            ax.text(column, row, str(matrix[row, column]), ha="center", va="center")
fig.suptitle("Confusion matrices")
fig.tight_layout()
fig.savefig(results_dir / "confusion_matrices.png", dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
for label, run, color in [
    ("LoRA", trainer, "#287f8e"),
    ("Full fine-tuning", trainer_base, "#bd6653"),
]:
    for key, series_label, linestyle in [
        ("loss", "train", "-"),
        ("eval_loss", "validation", "--"),
    ]:
        points = [
            (entry["step"], entry[key])
            for entry in run.state.log_history
            if key in entry and "step" in entry
        ]
        if points:
            steps, values = zip(*points)
            ax.plot(steps, values, color=color, linestyle=linestyle, label=f"{label} {series_label}")
ax.set_xlabel("Training step")
ax.set_ylabel("Loss")
ax.set_title("Training and validation loss")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(results_dir / "loss_curves.png", dpi=160, bbox_inches="tight")
plt.show()

lora_trainable = sum(parameter.numel() for parameter in lora_model.parameters() if parameter.requires_grad)
full_trainable = sum(parameter.numel() for parameter in base_model.parameters() if parameter.requires_grad)
parameter_counts = pd.Series(
    {"LoRA": lora_trainable, "Full fine-tuning": full_trainable}
)

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.barh(parameter_counts.index, parameter_counts.values, color=["#287f8e", "#bd6653"])
ax.set_xscale("log")
ax.set_xlabel("Trainable parameters (log scale)")
ax.set_title("Trainable parameter budget")
for bar, value in zip(bars, parameter_counts.values):
    ax.annotate(f"{value:,}", (value, bar.get_y() + bar.get_height() / 2), xytext=(6, 0), textcoords="offset points", va="center")
fig.tight_layout()
fig.savefig(results_dir / "trainable_parameters.png", dpi=160, bbox_inches="tight")
plt.show()

print(f"LoRA trains {lora_trainable:,} parameters; full fine-tuning trains {full_trainable:,}.")